# Finding moves

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimlab-wustl/Nimbus/blob/main/docs/tutorials/03_finding_moves.ipynb)

When no move builder keeps a constraint, the moves can be found from the feasible states alone: list the transitions
between them, group the transitions that one spin transformation reproduces into moves, and pick a backbone that
connects all states. The example is a constraint the package has no moves for: **Latin squares of order 3**. Every
cell of a 3 x 3 grid holds one of three symbols, and every row and every column holds each symbol once.

In [ ]:
# In Google Colab, fetch the repository and install the package; locally, run from the repository.
import os, sys
if "google.colab" in sys.modules and not os.path.exists("Nimbus"):
    !git clone -q https://github.com/aimlab-wustl/Nimbus.git
    !pip install -q -e Nimbus
if os.path.exists("Nimbus"):
    os.chdir("Nimbus")
elif os.path.basename(os.getcwd()) == "tutorials":
    os.chdir("../..")                                   # docs/tutorials -> repository root

## 1. The feasible states

Spin `9 r + 3 c + v` is +1 when cell `(r, c)` holds symbol `v`, so a square is 27 spins. There are 12 squares.

In [ ]:
import itertools

import numpy as np


def latin_squares(P=3):
    out = []
    for square in itertools.product(itertools.permutations(range(P)), repeat=P):
        if all(len({square[r][c] for r in range(P)}) == P for c in range(P)):
            s = -np.ones(P * P * P, dtype=np.int8)
            for r in range(P):
                for c in range(P):
                    s[(r * P + c) * P + square[r][c]] = 1
            out.append(s)
    return np.array(out)


def show(s, P=3):
    """The square of a spin state, one row per line."""
    grid = np.argmax(s.reshape(P, P, P), axis=2)
    return "\n".join(" ".join("ABC"[v] for v in row) for row in grid)


states = latin_squares()
print(len(states), "squares of", states.shape[1], "spins")
print(show(states[0]))

12 squares of 27 spins
A B C
B C A
C A B


## 2. Group the transitions into moves

`group_transitions` takes the feasible states and nothing else.

In [ ]:
from nimbus.transitions.grouping import group_transitions

grouping = group_transitions(states)
print(grouping.summary())

12 feasible states, 54 transitions (12 spins: 54)
9 moves; backbone [0, 8] (connects all states)
  move 0 (backbone): exchange s1<->s2, s3<->s6, s4<->s8, s5<->s7, s10<->s11, s12<->s15, s13<->s17, s14<->s16, s19<->s20, s21<->s24, s22<->s26, s23<->s25, 6 transitions
  move 1 (shortcut): exchange s1<->s2, s4<->s5, s7<->s8, s9<->s18, s10<->s20, s11<->s19, s12<->s21, s13<->s23, s14<->s22, s15<->s24, s16<->s26, s17<->s25, 6 transitions
  move 2 (shortcut): exchange s1<->s2, s4<->s5, s7<->s8, s10<->s11, s13<->s14, s16<->s17, s19<->s20, s22<->s23, s25<->s26, 6 transitions
  move 3 (shortcut): exchange s0<->s1, s3<->s4, s6<->s7, s9<->s10, s12<->s13, s15<->s16, s18<->s19, s21<->s22, s24<->s25, 6 transitions
  move 4 (shortcut): exchange s0<->s1, s3<->s4, s6<->s7, s9<->s19, s10<->s18, s11<->s20, s12<->s22, s13<->s21, s14<->s23, s15<->s25, s16<->s24, s17<->s26, 6 transitions
  move 5 (shortcut): exchange s0<->s1, s3<->s7, s4<->s6, s5<->s8, s9<->s10, s12<->s16, s13<->s15, s14<->s17, s18<->s19, s21<

Every move is a permutation of the spins: the transitions are grouped by the transformation that produces them.
Two of the moves form a backbone that connects all 12 squares; the others are shortcut candidates.

## 3. What a move does

Apply each backbone move to the first square.

In [ ]:
for r in grouping.backbone:
    move = grouping.moves[r].descriptor
    print(f"move {r}:")
    print(show(move.apply(states[0])), end="\n\n")

move 0:
A B C
C A B
B C A

move 8:
C B A
B A C
A C B



Move 8 exchanges the symbols A and C everywhere; move 0 exchanges the last two columns together with the symbols B
and C. A move found this way often has such a plain meaning. Once the meaning is clear, the same operation can be
written for a square of any size, without listing its states.

## 4. Check the machine

`verify_library` checks that the backbone keeps every square feasible and connects all of them. The compiled
machine must then give every move neuron half the energy change of its move, on every square, for any objective.

In [ ]:
from nimbus.compiler import compile_model
from nimbus.core.model import ObjectiveStructure, bind_weights
from nimbus.transitions.verify import verify_library

library = grouping.library("backbone")
print("feasible and connected:", verify_library(library, states).ok)

rng = np.random.default_rng(0)
terms = [tuple(sorted(rng.choice(27, size=int(rng.integers(1, 4)), replace=False).tolist())) for _ in range(12)]
weights = rng.normal(size=len(terms))
model = compile_model(ObjectiveStructure(n_spins=27, terms=terms), library, identities=grouping.identities)
binding = bind_weights(model, weights, 0.0)


def energy(S):
    return -np.array([sum(w * np.prod(s[list(t)]) for t, w in zip(terms, weights)) for s in S])


v = model.inputs(model.clause_values(states), binding)
dE = np.stack([energy(np.array([model.apply_action(r, s) for s in states])) - energy(states)
               for r in range(model.n_groups)], axis=1)
print(f"{model.n_groups} move neurons, {model.n_clauses} stored clauses; 2 v = dE:", np.allclose(2 * v, dE))

feasible and connected: True
2 move neurons, 50 stored clauses; 2 v = dE: True


## 5. When no move exists

Select 3 of the 8 vertices of a cycle, no two adjacent. Moving a selected vertex one step is allowed only when the
vertex it moves next to is also unselected: a condition on two spins at once, which no single product of spins
expresses. The transitions are reported as ungrouped instead of being forced into moves.

In [ ]:
def independent_sets_on_cycle(n, k):
    out = []
    for sel in itertools.combinations(range(n), k):
        if all((b - a) % n not in (1, n - 1) for a, b in itertools.combinations(sel, 2)):
            s = -np.ones(n, dtype=np.int8)
            s[list(sel)] = 1
            out.append(s)
    return np.array(out)


print(group_transitions(independent_sets_on_cycle(8, 3)).summary())

16 feasible states, 36 transitions (2 spins: 36)
0 moves; backbone [] (does NOT connect all states)
36 transitions are in no move


## Next

* Use the moves in a problem: return `grouping.library("backbone")` from the `transition_library` of a
  [problem adapter](https://aimlab-wustl.github.io/Nimbus/guides/new_problem/adapter/).
* Moves written from the structure of a problem, and the four conditions every move meets:
  [Your own moves](https://aimlab-wustl.github.io/Nimbus/guides/new_problem/own_moves/).
* Which moves the machine can use, and which it cannot: [Which moves work](https://aimlab-wustl.github.io/Nimbus/concepts/moves/).